In [10]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from datasets import Dataset


model_id = "Hailey97/best_model"

print(f"Loading model from Hugging Face: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)



Loading model from Hugging Face: Hailey97/best_model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [11]:

test_file = "test_data.xlsx"

try:
    df_test = pd.read_excel(test_file)
    print(f"Successfully loaded {test_file}")
except FileNotFoundError:
    print(f"Error: Not found {test_file}。")

# ==========================================
# 1. Preprocesses the test data
# ==========================================

texts = df_test['text'].astype(str).tolist()
test_dataset = Dataset.from_dict({"text": texts})

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_ds = test_dataset.map(tokenize_function, batched=True)

# ==========================================
# 2. Generates predictions
# ==========================================
print("Executing model inference…")
trainer = Trainer(model=model)
raw_predictions = trainer.predict(tokenized_ds)

# Use Sigmoid to convert the output into probabilities, and use 0.5 as the threshold to convert them into 1/0.
probs = torch.sigmoid(torch.from_numpy(raw_predictions.predictions)).numpy()
y_pred = (probs > 0.5).astype(int)

# ==========================================
# 3. Saves to output.xlsx
# ==========================================

topic_columns = ['bathroom-atmosphere', 'bathroom-cleanliness',
       'bathroom-facilities', 'bathroom-general', 'bed-cleanliness',
       'bed-general', 'catering-general', 'catering-price', 'hotel-atmosphere',
       'hotel-cleanliness', 'hotel-facilities', 'hotel-general',
       'hotel-location', 'hotel-price', 'parking', 'room-atmosphere',
       'room-cleanliness', 'room-facilities', 'room-general', 'staff']

# Fill the predicted values into the corresponding 20 topic columns in the DataFrame.
# This will ensure that the structure of output.xlsx is exactly the same as test_data.xlsx.
df_test[topic_columns] = y_pred

output_file = "output.xlsx"
df_test.to_excel(output_file, index=False)

print(f"Completed! The prediction results have been saved to {output_file}")

Successfully loaded test_data.xlsx


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Executing model inference…


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Completed! The prediction results have been saved to output.xlsx
